# EEG_16 — Clustering Soggetti per Matrice di Connettività

**Direzione**: Francesco Pelosin — approccio data-first (non accuracy-first)

**Logica**:
1. Calcola matrice di connettività media per soggetto (5 metriche: pcc, abs_pcc, im_pcc, wpli, plv)
2. Clustering soggetti con K-means (k=2,3,4) → silhouette per scegliere k ottimale
3. Matrice media per cluster → confronto visivo inter-cluster
4. Post-hoc: overlay accuracy EEG_12/13 → i cluster EEG predicono la performance?

**Riferimento**: EEG_08b mostra variabilità inter-soggetto dominante (ε²=0.85). Questo notebook trova la struttura.

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
from scipy.signal import hilbert
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg16')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg16'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'

METRICS        = ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
K_LIST         = [2, 3, 4]       # numero cluster da esplorare
N_INIT_KMEANS  = 20              # riavvii K-means per stabilità
RANDOM_SEED    = 42

# Carica accuracy EEG_12/13 se disponibile (post-hoc)
ACC_FILE_12 = project_root / 'figures' / 'eeg12_subject_ranking.csv'
ACC_FILE_13 = project_root / 'figures' / 'eeg13b_subject_ranking.csv'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}

# Indice dei trial per metrica → {sid: [path, ...]}
def build_index(metric):
    root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
    idx = defaultdict(list)
    if not root.exists():
        log.warning(f'Directory non trovata: {root}')
        return idx
    for p in sorted(root.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m:
            idx[int(m.group(1))].append(p)
    return idx

# Usa abs_pcc come metrica principale per la lista soggetti
MAIN_IDX = build_index('abs_pcc')
ALL_SUBJ = sorted(MAIN_IDX.keys())
log.info(f'Soggetti trovati: {len(ALL_SUBJ)}')
log.info(f'Metriche: {METRICS}')

## §2 — Calcolo Matrici di Connettività Per-Soggetto

Per ogni soggetto e ogni metrica: media delle matrici trial-level → (61,61) fingerprint.

- **PCC**: `corrcoef(x)` — correlazione lineare
- **abs_PCC**: `|PCC|` — correlazione assoluta (topologia principale del progetto)
- **im_PCC**: parte immaginaria della coerenza spettrale → robusta agli artefatti di volume conduction
- **wPLI**: weighted Phase Lag Index → sincronizzazione di fase pesata
- **PLV**: Phase Locking Value → accoppiamento di fase puro

In [ ]:
# ── Funzioni di connettività ──────────────────────────────────────────────────

def compute_pcc(x):
    """x: (61, T) → (61, 61) PCC matrix"""
    return np.corrcoef(x)

def compute_abs_pcc(x):
    return np.abs(np.corrcoef(x))

def compute_im_pcc(x):
    """Imaginary part of cross-spectrum (volume conduction robust)."""
    an = hilbert(x, axis=1)                    # (61, T) analytic signal
    # cross-spectrum matrix (mean over time)
    cs = an @ np.conj(an.T) / x.shape[1]      # (61, 61)
    return np.imag(cs)

def compute_wpli(x):
    """Weighted Phase Lag Index — vectorized."""
    an = hilbert(x, axis=1)                    # (61, T)
    # pairwise cross-spectrum imaginary part
    # Im(X_i * conj(X_j)) for all i,j simultaneously
    cs = an[:, None, :] * np.conj(an[None, :, :])   # (61, 61, T)
    im_cs = np.imag(cs)                              # (61, 61, T)
    num = np.abs(np.mean(im_cs, axis=2))             # |E[Im]|
    den = np.mean(np.abs(im_cs), axis=2) + 1e-12    # E[|Im|]
    return num / den

def compute_plv(x):
    """Phase Locking Value — vectorized."""
    an   = hilbert(x, axis=1)                  # (61, T)
    ph   = np.angle(an)                        # (61, T) phases
    # phase difference for all pairs
    dphi = ph[:, None, :] - ph[None, :, :]    # (61, 61, T)
    plv  = np.abs(np.mean(np.exp(1j * dphi), axis=2))  # (61, 61)
    return plv

CONN_FN = {
    'pcc':     compute_pcc,
    'abs_pcc': compute_abs_pcc,
    'im_pcc':  compute_im_pcc,
    'wpli':    compute_wpli,
    'plv':     compute_plv,
}

# ── Cache per-soggetto ────────────────────────────────────────────────────────
CONN_CACHE = CKPT_DIR / 'subject_conn_matrices.npz'

if CONN_CACHE.exists():
    log.info(f'Cache trovata: {CONN_CACHE}')
    _c = np.load(CONN_CACHE, allow_pickle=True)
    CONN = _c['conn'].item()   # dict: metric → (n_subj, 61, 61)
    SUBJ_IDS = _c['subj_ids'].tolist()
else:
    log.info('Calcolo matrici di connettività per-soggetto...')
    CONN     = {m: [] for m in METRICS}
    SUBJ_IDS = []

    for sid in tqdm(ALL_SUBJ, desc='Soggetti'):
        paths = MAIN_IDX[sid]
        if len(paths) < 10:
            continue

        # Accumula matrici trial-level per ogni metrica
        accum = {m: [] for m in METRICS}
        for p in paths:
            try:
                d = torch.load(p, weights_only=False)
                x = d['x'].float().numpy()   # (61, 384)
                if x.shape != (N_CHANNELS, N_SAMPLES):
                    continue
                for m in METRICS:
                    accum[m].append(CONN_FN[m](x))
            except Exception:
                continue

        if len(accum['abs_pcc']) < 5:
            continue

        for m in METRICS:
            CONN[m].append(np.mean(accum[m], axis=0))   # (61, 61)
        SUBJ_IDS.append(sid)

    for m in METRICS:
        CONN[m] = np.stack(CONN[m])   # (n_subj, 61, 61)

    np.savez(CONN_CACHE, conn=CONN, subj_ids=np.array(SUBJ_IDS))
    log.info(f'Salvato: {CONN_CACHE}')

N_SUBJ = len(SUBJ_IDS)
log.info(f'Soggetti validi: {N_SUBJ}')
for m in METRICS:
    log.info(f'  {m}: {CONN[m].shape}')

## §3 — Clustering K-Means + Silhouette

Per ogni metrica: vettorizza triangolo superiore → PCA → K-means con k ∈ {2,3,4}.
Silhouette score per selezionare k ottimale.

In [ ]:
def upper_tri(mat):
    """(61,61) → vettore triangolo superiore senza diagonale (1830D)"""
    idx = np.triu_indices(mat.shape[0], k=1)
    return mat[idx]

def vectorize_conn(conn_3d):
    """(n_subj, 61, 61) → (n_subj, 1830)"""
    return np.stack([upper_tri(conn_3d[i]) for i in range(len(conn_3d))])

CLUSTER_RESULTS = {}   # metric → {'labels_k': {k: array}, 'sil': {k: float}, 'feat_pca': array}

fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 4), sharey=True)
fig.suptitle('Silhouette Score per Metrica e k', fontsize=13, fontweight='bold')

for ax, metric in zip(axes, METRICS):
    X_raw = vectorize_conn(CONN[metric])           # (n_subj, 1830)
    X_sc  = StandardScaler().fit_transform(X_raw)  # normalizza feature

    # PCA a 20 componenti per clustering stabile
    n_comp = min(20, X_sc.shape[0] - 1, X_sc.shape[1])
    pca    = PCA(n_components=n_comp, random_state=RANDOM_SEED)
    X_pca  = pca.fit_transform(X_sc)               # (n_subj, 20)
    var_exp = pca.explained_variance_ratio_.sum()

    sil_scores = {}
    labels_k   = {}
    for k in K_LIST:
        km = KMeans(n_clusters=k, n_init=N_INIT_KMEANS, random_state=RANDOM_SEED)
        labels = km.fit_predict(X_pca)
        sil    = silhouette_score(X_pca, labels) if len(set(labels)) > 1 else 0.0
        sil_scores[k] = sil
        labels_k[k]   = labels
        log.info(f'{metric}  k={k}  silhouette={sil:.4f}')

    CLUSTER_RESULTS[metric] = {
        'labels_k': labels_k,
        'sil':      sil_scores,
        'feat_pca': X_pca,
        'var_exp':  float(var_exp),
    }

    ax.bar([str(k) for k in K_LIST], [sil_scores[k] for k in K_LIST],
           color=['#2C7BB6', '#1A9641', '#D7191C'], alpha=0.85)
    ax.set_title(f'{metric}\n(PCA var={var_exp:.1%})', fontsize=10)
    ax.set_xlabel('k'); ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 0.6)

axes[0].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg16_silhouette.png', dpi=150)
plt.show()
log.info('Silhouette plot salvato')

## §4 — Visualizzazione Cluster (PCA 2D)

Scatter plot 2D per ogni metrica e ogni k. Colori = cluster.

In [ ]:
COLORS = ['#2C7BB6', '#D7191C', '#1A9641', '#FF7F00']
MARKERS = ['o', 's', '^', 'D']

fig, axes = plt.subplots(len(K_LIST), len(METRICS),
                         figsize=(4 * len(METRICS), 3.5 * len(K_LIST)))
fig.suptitle('Cluster Soggetti per Connettività (PCA 2D)', fontsize=13, fontweight='bold')

for col, metric in enumerate(METRICS):
    X_pca = CLUSTER_RESULTS[metric]['feat_pca']   # (n_subj, 20)
    # proietta su prime 2 PC per visualizzazione
    X_2d  = X_pca[:, :2]

    for row, k in enumerate(K_LIST):
        ax     = axes[row][col]
        labels = CLUSTER_RESULTS[metric]['labels_k'][k]
        sil    = CLUSTER_RESULTS[metric]['sil'][k]

        for c in range(k):
            mask = labels == c
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                       c=COLORS[c], marker=MARKERS[c],
                       s=60, alpha=0.8, label=f'C{c} (n={mask.sum()})',
                       edgecolors='white', linewidths=0.5)
        # label soggetti
        for i, sid in enumerate(SUBJ_IDS):
            ax.annotate(f'P{sid:03d}', (X_2d[i, 0], X_2d[i, 1]),
                        fontsize=4, alpha=0.6,
                        xytext=(2, 2), textcoords='offset points')

        ax.set_title(f'{metric} | k={k} | sil={sil:.3f}', fontsize=9)
        ax.legend(fontsize=7, markerscale=0.8)
        ax.set_xlabel('PC1', fontsize=8); ax.set_ylabel('PC2', fontsize=8)
        ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg16_clusters_2d.png', dpi=150)
plt.show()

## §5 — Matrice Media di Connettività per Cluster

Per ogni metrica × k ottimale: heatmap della matrice media per cluster.
Permette confronto visivo: i cluster hanno topologie di connettività diverse?

In [ ]:
# k ottimale per metrica = argmax silhouette
BEST_K = {m: max(K_LIST, key=lambda k: CLUSTER_RESULTS[m]['sil'][k])
          for m in METRICS}
log.info('k ottimale per metrica: ' + str({m: BEST_K[m] for m in METRICS}))

for metric in METRICS:
    k      = BEST_K[metric]
    labels = CLUSTER_RESULTS[metric]['labels_k'][k]
    conn   = CONN[metric]   # (n_subj, 61, 61)
    sil    = CLUSTER_RESULTS[metric]['sil'][k]

    fig, axes = plt.subplots(1, k + 1, figsize=(4 * (k + 1), 4))
    fig.suptitle(f'{metric} — Matrici medie per cluster (k={k}, sil={sil:.3f})',
                 fontsize=12, fontweight='bold')

    # matrice globale (tutti i soggetti) come riferimento
    global_mean = conn.mean(axis=0)
    vmin = global_mean.min(); vmax = global_mean.max()

    im0 = axes[0].imshow(global_mean, cmap='RdBu_r', vmin=vmin, vmax=vmax)
    axes[0].set_title(f'Media globale\n(N={len(labels)})', fontsize=10)
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    for c in range(k):
        mask     = labels == c
        mean_mat = conn[mask].mean(axis=0)
        ax       = axes[c + 1]
        im = ax.imshow(mean_mat, cmap='RdBu_r', vmin=vmin, vmax=vmax)
        ax.set_title(f'Cluster {c}\n(N={mask.sum()})', fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    fname = FIG_DIR / f'eeg16_conn_cluster_{metric}_k{k}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    log.info(f'Salvato: {fname}')

## §6 — Dendrogram Gerarchico

Clustering gerarchico (Ward linkage) su abs_pcc: mostra la struttura naturale dei soggetti.

In [ ]:
for metric in ['abs_pcc', 'wpli']:   # le due metriche principali
    X_raw = vectorize_conn(CONN[metric])
    X_sc  = StandardScaler().fit_transform(X_raw)

    Z = linkage(X_sc, method='ward')

    fig, ax = plt.subplots(figsize=(max(12, N_SUBJ * 0.3), 5))
    dendrogram(Z,
               labels=[f'P{sid:03d}' for sid in SUBJ_IDS],
               ax=ax, leaf_rotation=90, leaf_font_size=7,
               color_threshold=0.7 * max(Z[:, 2]))
    ax.set_title(f'Clustering gerarchico soggetti — {metric} (Ward)', fontsize=12)
    ax.set_ylabel('Distanza Ward')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'eeg16_dendrogram_{metric}.png', dpi=150)
    plt.show()

## §7 — Post-hoc: Overlay Accuracy

I cluster EEG riflettono la capacità di imagined speech?
Carica risultati EEG_12 (W-HGNN subject-specific) e overlay sui cluster.

In [ ]:
# Carica accuracy disponibili
acc_data = {}
for label, fpath in [('EEG_12', ACC_FILE_12), ('EEG_13', ACC_FILE_13)]:
    if fpath.exists():
        df = pd.read_csv(fpath)
        # normalizza colonna Subject: P000 → 0
        if 'Subject' in df.columns:
            df['sid'] = df['Subject'].str.extract(r'(\d+)').astype(int)
        if 'Test bAcc' in df.columns:
            acc_data[label] = dict(zip(df['sid'], df['Test bAcc']))
        elif 'test_bacc' in df.columns:
            acc_data[label] = dict(zip(df['sid'], df['test_bacc']))
        log.info(f'Caricato {label}: {len(acc_data[label])} soggetti')
    else:
        log.warning(f'{label} non trovato: {fpath}')

if not acc_data:
    print('[INFO] Nessun file accuracy trovato — esegui EEG_12/13 prima.')
else:
    chance = 1 / N_CLASSES

    for acc_label, acc_dict in acc_data.items():
        for metric in ['abs_pcc', 'wpli']:
            k      = BEST_K[metric]
            labels = CLUSTER_RESULTS[metric]['labels_k'][k]
            X_pca  = CLUSTER_RESULTS[metric]['feat_pca'][:, :2]

            # accuracy per soggetto nell'ordine SUBJ_IDS
            accs = np.array([acc_dict.get(sid, np.nan) for sid in SUBJ_IDS])

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle(f'Post-hoc: {acc_label} accuracy × cluster {metric} (k={k})',
                         fontsize=12, fontweight='bold')

            # ── Scatter: cluster color + accuracy marker size ──────────────
            ax = axes[0]
            valid = ~np.isnan(accs)
            sc = ax.scatter(X_pca[valid, 0], X_pca[valid, 1],
                            c=accs[valid], cmap='RdYlGn',
                            s=120, vmin=chance, vmax=accs[valid].max(),
                            edgecolors=[COLORS[labels[i]] for i in np.where(valid)[0]],
                            linewidths=2.5)
            plt.colorbar(sc, ax=ax, label='bAcc')
            ax.axhline(0, color='k', alpha=0.2); ax.axvline(0, color='k', alpha=0.2)
            ax.set_title('Scatter: fill=accuracy, bordo=cluster'); ax.grid(alpha=0.2)
            for i in np.where(valid)[0]:
                ax.annotate(f'P{SUBJ_IDS[i]:03d}', (X_pca[i, 0], X_pca[i, 1]),
                            fontsize=5, xytext=(2, 2), textcoords='offset points')

            # ── Box plot accuracy per cluster ──────────────────────────────
            ax2 = axes[1]
            cluster_accs = [accs[labels == c] for c in range(k)]
            cluster_accs_valid = [a[~np.isnan(a)] for a in cluster_accs]
            bp = ax2.boxplot(cluster_accs_valid,
                             patch_artist=True, notch=False,
                             medianprops=dict(color='black', linewidth=2))
            for patch, c in zip(bp['boxes'], range(k)):
                patch.set_facecolor(COLORS[c]); patch.set_alpha(0.7)
            ax2.axhline(chance, color='k', linestyle='--', linewidth=1.5,
                        label=f'Chance ({chance:.0%})')
            ax2.set_xticklabels([f'Cluster {c}\n(n={len(cluster_accs_valid[c])})'
                                  for c in range(k)])
            ax2.set_ylabel('Balanced Accuracy'); ax2.legend()
            ax2.set_title('Distribuzione accuracy per cluster'); ax2.grid(axis='y', alpha=0.3)

            # Stampa medie
            for c in range(k):
                cv = cluster_accs_valid[c]
                if len(cv) > 0:
                    print(f'{metric} k={k} Cluster {c}: mean={cv.mean():.4f} ± {cv.std():.4f}  (n={len(cv)})')

            plt.tight_layout()
            fname = FIG_DIR / f'eeg16_posthoc_{acc_label}_{metric}_k{k}.png'
            plt.savefig(fname, dpi=150)
            plt.show()
            log.info(f'Salvato: {fname}')

## §8 — Riepilogo Cluster

Tabella: soggetto → cluster per ogni metrica (k ottimale).

In [ ]:
rows = []
for i, sid in enumerate(SUBJ_IDS):
    row = {'Subject': f'P{sid:03d}'}
    for m in METRICS:
        k = BEST_K[m]
        row[f'cluster_{m}'] = CLUSTER_RESULTS[m]['labels_k'][k][i]
    # aggiungi accuracy se disponibile
    for acc_label, acc_dict in acc_data.items():
        row[f'acc_{acc_label}'] = round(acc_dict.get(sid, np.nan), 4)
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.to_csv(FIG_DIR / 'eeg16_subject_clusters.csv', index=False)
print(df_summary.to_string(index=False))

# Concordanza tra metriche
print('\n── k ottimali ──')
for m in METRICS:
    k   = BEST_K[m]
    sil = CLUSTER_RESULTS[m]['sil'][k]
    print(f'  {m}: k={k}  silhouette={sil:.4f}')